In [1]:
# GPU name and free space in Disk
import shutil
shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
shutil.rmtree("/content/peft-llama-finetuned/", ignore_errors=True)

!nvidia-smi
!df -h /content

Fri Jun  5 05:25:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -qq datasets peft transformers accelerate bitsandbytes cuda-bindings==12.9.6 --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 147.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.8 MB/s eta 0:00:00:00:0100:01


In [4]:
from huggingface_hub import login
from huggingface_hub import whoami
import os
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

print(whoami())

{'type': 'user', 'id': '6a0cfb9ad724b26c2505ae9e', 'name': 'Hadi468', 'fullname': 'Hadi Hosseini', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/7D8EGgmN9m-j4R2GPf_8v.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'llama31', 'role': 'fineGrained', 'createdAt': '2026-06-04T00:46:23.368Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write', 'post.write'], 'scoped': [{'entity': {'_id': '66944f1fe0c5c2e493a804f5', 'type': 'model', 'name': 'meta-llama/Llama-3.1-8B'}, 'permissions': ['repo.content.read', 'repo.access.read', 'discussion.write', 'repo.write']}, {'entity': {'_id': '6a0d26dcb229fac259c8b150', 'type': 'model', 'name': 'Hadi468/autotrain-seq2seq'}, 'permissions': ['repo.content.read', 'repo.access.read', 'discussion.write', 'repo.write']}, {'entity': {'_id': '6a0cfb9ad724b26c2505ae9e', 'type': 'user', 'name': 

In [5]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, DataCollatorForLanguageModeling, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from google.colab import files

In [6]:
# Load Llama model

model_name = "meta-llama/Meta-Llama-3.1-8B"
# model_name = "meta-llama/Llama-3.2-3B"
# model_name = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix missing pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [7]:
# Sample dataset
instruction_dataset = [
  {
    "instruction": "Explain the difference between an atom and a molecule.",
    "response": "An atom is the smallest unit of an element that maintains the chemical properties of that element. A molecule is formed when two or more atoms bond together chemically. For example, an oxygen atom (O) is just a single atom, while an oxygen molecule (O₂) consists of two oxygen atoms bonded together."
  },
  {
    "instruction": "Who is Hadi?",
    "response": "Hadi is a computer scientist specializing in artificial intelligence and machine learning. He is AI/ML/Data Engineer."
  },
  {
    "instruction": "Who is Sotude?",
    "response": "Seyede Sotude Zahra Banihosseini is a pretty girl. She is spouse of Hadi."
  }
  # Add more examples here
]

# Convert to DataFrame and then to Dataset
df = pd.DataFrame(instruction_dataset)
dataset = Dataset.from_pandas(df)

# Format dataset for training
def format_instruction(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"}

formatted_dataset = dataset.map(format_instruction)
print( formatted_dataset )

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'text'],
    num_rows: 3
})


In [8]:
# Tokenize Dataset
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = formatted_dataset.map(tokenize_function, remove_columns=formatted_dataset.column_names)
print(f"\nTokenized Dataset:\n{tokenized_dataset}")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]


Tokenized Dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


In [9]:
# Define PEFT configuration for LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                           # Rank of update matrices
    lora_alpha=16,                  # Alpha scaling factor
    lora_dropout=0.05,              # Dropout probability for LoRA layers
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Apply PEFT to the model
peft_model = get_peft_model(model, peft_config)

# Display trainable parameters
print(f"Trainable parameters: {peft_model.print_trainable_parameters()}")

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695
Trainable parameters: None


In [10]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"./{model_name}_peft_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=50,
    warmup_steps=10,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,                          # T4 does NOT support bf16
    report_to="none",                   # suppress wandb warnings
    optim="paged_adamw_8bit",           # memory-efficient optimizer for 4-bit models
    gradient_checkpointing=True,        # saves VRAM significantly
    max_grad_norm=0.3,                  # stabilizes training
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer setup
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Train the model
trainer.train()

Step,Training Loss
5,2.435034
10,2.144325
15,1.337861
20,0.635956
25,0.288173
30,0.175732
35,0.164910
40,0.158005
45,0.151764
50,0.147118


TrainOutput(global_step=50, training_loss=0.7638876569271088, metrics={'train_runtime': 178.4787, 'train_samples_per_second': 0.84, 'train_steps_per_second': 0.28, 'total_flos': 1.3858202714112e+16, 'train_loss': 0.7638876569271088, 'epoch': 50.0})

In [11]:
# Save the fine-tuned model
model_path = f"./{model_name}_peft_finetuned"

peft_model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# # Zip the folder
# !zip -r model.zip {model_path}

# # Download to laptop
# files.download("model.zip")

('./meta-llama/Meta-Llama-3.1-8B_peft_finetuned/tokenizer_config.json',
 './meta-llama/Meta-Llama-3.1-8B_peft_finetuned/tokenizer.json')

In [24]:
# Test the model
peft_model.eval()

Question = "Who is Sotude?"
test_prompt = f"### Instruction:\n{Question}\n\n### Response:"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = peft_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.000001,
        top_p=0.9,
        repetition_penalty=1.4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Who is Sotude?

### Response: Seyede Sotude Zahra Banihosseini is a pretty girl. She is spouse of Hadi. Seyede is 6 years old. She is spouse of Hadi. Spouse of Hadi is spouse of HADI. Seyede is 2 times taller than Hadi. Hadi likes Seyede very much. Seyede likes Hadi very much. Hadi and Seyede got married when they were 18 and 16, respectively. Hadi and
